In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, balanced_accuracy_score


# ── 1. Load data ─────────────────────────────────────────────────
df_bind = pd.read_excel('../data/single_metal_atoms_on_graphene_binding_energy_and_diffusion_barrier.xlsx')
df_bind_g = df_bind.groupby('Metal')[['Binding', 'Diffusion']].mean().reset_index()

df_aff = pd.read_excel("../data/supported_metal_M_oxygen_affinity_QMO_and_support_metal_affinity_QMM_prime.xlsx")
df_aff = df_aff.rename(columns={df_aff.columns[1]: 'QMO'})

df_nps = pd.read_excel('./NPs.xlsx')
df_nps = df_nps.rename(columns={'反应后MOF表面是否有纳米粒子': 'ExternalNP'})
df_nps['MOF'] = df_nps['MOF'].ffill()

# Define your renaming dictionary
mof_rename_dict = {
}

# Apply the replacement
df_nps["MOF"] = df_nps["MOF"].replace(mof_rename_dict)

df_mof = pd.read_csv('../data/MOF_factor.csv').reset_index(drop = True)
# Optional: check for consistency with df_mof
for i in df_nps["MOF"]:
    assert i in list(df_mof["MOF"]), f"{i} not in df_mof"
# ── 2. Merge all descriptors ─────────────────────────────────────
df = (
    df_nps
    .merge(df_mof, on='MOF', how='left')
    .merge(df_bind_g.rename(columns={'Binding': 'BindingEnergy', 'Diffusion': 'DiffusionBarrier'}),
           left_on='M', right_on='Metal', how='left')
    .merge(df_aff[['Metal', 'QMO']], left_on='M', right_on='Metal', how='left')
    .drop(columns=['Metal', 'Metal_aff'], errors='ignore')
)

df = df.dropna(subset=['BindingEnergy', 'DiffusionBarrier', 'QMO'])
df = df.drop(columns=['Metal_x', 'Metal_y'], errors='ignore')

noble_set = {'Au', 'Ag', 'Pt', 'Pd', 'Ir', 'Rh', 'Ru'}
df['Noble'] = df['M'].apply(lambda x: 1 if x in noble_set else 0)

core_feats = ['BindingEnergy','DiffusionBarrier','QMO','Noble']
extra_feats = [f"Factor{i}" for i in range(1, 46)]

features_all = core_feats + extra_feats

# Drop rows with missing values
df = df.dropna(subset=features_all + ['ExternalNP'])

X = df[features_all].copy()
y = df['ExternalNP'].astype(int).copy()

# ── 2. Train & evaluate logistic ────────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(random_state = 42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
bal_acc_scores = cross_val_score(
    pipeline, X, y, cv=cv,
    scoring=make_scorer(balanced_accuracy_score)
)

print(f"5-fold Accuracy: {acc_scores.mean():.3f} ± {acc_scores.std():.3f}")
print(f"5-fold Balanced Accuracy: {bal_acc_scores.mean():.3f} ± {bal_acc_scores.std():.3f}")


# ── 2. Train & evaluate MLP (8-4) ────────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(10,4), max_iter=100000, random_state=42, tol = 1e-6, activation="tanh"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
bal_acc_scores = cross_val_score(
    pipeline, X, y, cv=cv,
    scoring=make_scorer(balanced_accuracy_score)
)

print(f"5-fold Accuracy: {acc_scores.mean():.3f} ± {acc_scores.std():.3f}")
print(f"5-fold Balanced Accuracy: {bal_acc_scores.mean():.3f} ± {bal_acc_scores.std():.3f}")


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(10,4), max_iter=100000, random_state=42, tol = 1e-6, activation="tanh"))
])
# Re-fit on full dataset
pipeline.fit(X, y)

5-fold Accuracy: 0.818 ± 0.081
5-fold Balanced Accuracy: 0.817 ± 0.076
5-fold Accuracy: 0.782 ± 0.067
5-fold Balanced Accuracy: 0.776 ± 0.070


Pipeline(steps=[('scaler', StandardScaler()),
                ('mlp',
                 MLPClassifier(activation='tanh', hidden_layer_sizes=(10, 4),
                               max_iter=100000, random_state=42, tol=1e-06))])

In [3]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score

# --------------------------------------------------
# settings
# --------------------------------------------------
metal_col = "M"   # change if your metal column has another name
target_col = "ExternalNP"

feats_without_f37 = ['DiffusionBarrier', 'QMO', 'Noble','BindingEnergy']
feats_with_f37    = ['DiffusionBarrier', 'QMO', 'Noble', 'Factor37']

# use the SAME rows for both models
needed_cols = list(set(feats_without_f37 + feats_with_f37 + [target_col, metal_col]))
df_cmp = df.dropna(subset=needed_cols).copy()

X_without = df_cmp[feats_without_f37].copy()
X_with    = df_cmp[feats_with_f37].copy()
y = df_cmp[target_col].astype(int).copy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_pipeline():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=(10, 4),
            max_iter=100000,
            random_state=42,
            tol=1e-6,
            activation="tanh"
        ))
    ])

# --------------------------------------------------
# out-of-fold predictions for both models
# --------------------------------------------------
y_pred_without = cross_val_predict(
    make_pipeline(), X_without, y, cv=cv, method='predict'
)

y_pred_with = cross_val_predict(
    make_pipeline(), X_with, y, cv=cv, method='predict'
)

# store predictions
df_cmp = df_cmp.copy()
df_cmp["y_true"] = y.values
df_cmp["y_pred_without_f37"] = y_pred_without
df_cmp["y_pred_with_f37"] = y_pred_with

# --------------------------------------------------
# per-metal accuracy
# --------------------------------------------------
rows = []
for metal, g in df_cmp.groupby(metal_col):
    acc_without = accuracy_score(g["y_true"], g["y_pred_without_f37"])
    acc_with    = accuracy_score(g["y_true"], g["y_pred_with_f37"])
    n = len(g)

    rows.append({
        "Metal": metal,
        "n": n,
        "Accuracy_without_F37": acc_without,
        "Accuracy_with_F37": acc_with,
        "Delta_with_minus_without": acc_with - acc_without
    })

acc_by_metal = pd.DataFrame(rows).sort_values("Delta_with_minus_without", ascending=False)
print(acc_by_metal)

# --------------------------------------------------
# specifically print Ir and Pt
# --------------------------------------------------
for metal in ["Ir", "Pt"]:
    sub = acc_by_metal[acc_by_metal["Metal"] == metal]
    if len(sub) == 0:
        print(f"{metal}: not found in dataset")
    else:
        r = sub.iloc[0]
        print(
            f"{metal}: n={int(r['n'])}, "
            f"without Factor 37 = {r['Accuracy_without_F37']:.3f}, "
            f"with Factor 37 = {r['Accuracy_with_F37']:.3f}, "
            f"delta = {r['Delta_with_minus_without']:+.3f}"
        )

  Metal   n  Accuracy_without_F37  Accuracy_with_F37  Delta_with_minus_without
7    Pt  11              0.545455           0.727273                  0.181818
4    Ir  11              0.454545           0.636364                  0.181818
6    Pd  11              0.909091           1.000000                  0.090909
0    Ag  11              1.000000           1.000000                  0.000000
1    Au  11              1.000000           1.000000                  0.000000
2    Co  11              0.909091           0.909091                  0.000000
8    Rh  11              0.909091           0.818182                 -0.090909
3    Cu  11              0.909091           0.818182                 -0.090909
5    Ni  11              1.000000           0.909091                 -0.090909
9    Ru  11              1.000000           0.909091                 -0.090909
Ir: n=11, without Factor 37 = 0.455, with Factor 37 = 0.636, delta = +0.182
Pt: n=11, without Factor 37 = 0.545, with Factor 37 = 0